Importação das bibliotecas

In [ ]:
from pathlib import Path
import urllib.request
import zipfile
import pandas as pd

Carregar e usar os dados

In [ ]:
class CarregadorDados:

    def __init__(self, url, arquivos, pasta="data"):
        self.url = url
        self.arquivos = arquivos
        self.pasta = Path(pasta)
        self.pasta_raw = self.pasta / "raw"
        self.pasta_raw.mkdir(parents=True, exist_ok=True)

    def caminho_csv(self, arquivo):
        return self.pasta / (Path(arquivo).stem + ".csv")

    def baixar(self):
        zip_path = self.pasta / "occupancy_data.zip"
        urllib.request.urlretrieve(self.url, zip_path)

        with zipfile.ZipFile(zip_path) as z:
            z.extractall(self.pasta_raw)

        zip_path.unlink()

    def carregar(self):
        faltando = [
            a for a in self.arquivos
            if not self.caminho_csv(a).exists()
            and not (self.pasta_raw / a).exists()
        ]

        if faltando:
            print("Baixando:", faltando)
            self.baixar()

        dados = {}
        for arquivo in self.arquivos:
            csv = self.caminho_csv(arquivo)

            if not csv.exists():
                pd.read_csv(self.pasta_raw / arquivo).to_csv(csv, index=False)

            dados[csv.stem] = pd.read_csv(csv)

        return dados


carregador = CarregadorDados(
    url="https://archive.ics.uci.edu/ml/machine-learning-databases/00357/occupancy_data.zip",
    arquivos=["datatraining.txt", "datatest.txt", "datatest2.txt"],
    pasta="../data"
)

dfs = carregador.carregar()
df = dfs["datatest"]

print("Base de dados carregada com sucesso!")
df.head()